# Data Processing - The Simpsons Dataset

**Author(s):** write your name(s) here

This notebook performs the initial exploration, cleaning, merging and aggregation of the Simpsons scripts dataset.  
The resulting CSV files are prepared for the interactive visualizations in Altair and Streamlit.

## 1. Load raw datasets

We load the two raw datasets:

- `simpsons_script_lines.csv`: contains the dialogue lines.
- `simpsons_episodes.csv`: contains episode metadata such as season, episode number, title and ratings.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Make pandas output easier to inspect
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path(".")

lines_path = DATA_DIR / "simpsons_script_lines.csv"
episodes_path = DATA_DIR / "simpsons_episodes.csv"

lines = pd.read_csv(lines_path, low_memory=False)
episodes = pd.read_csv(episodes_path, low_memory=False)

print("Script lines shape:", lines.shape)
print("Episodes shape:", episodes.shape)

## 2. Exploratory Data Analysis

Before cleaning the data, we inspect the raw files to understand their structure, missing values and possible quality problems.

In [ ]:
# First rows of script lines
lines.head()

In [ ]:
# First rows of episodes
episodes.head()

In [ ]:
# Column names
print("Script lines columns:")
print(lines.columns.tolist())

print("\nEpisodes columns:")
print(episodes.columns.tolist())

In [ ]:
# Data types and non-null counts
print("Script lines info:")
lines.info()

print("\nEpisodes info:")
episodes.info()

In [ ]:
# Missing values in script lines
missing_lines = lines.isna().sum().sort_values(ascending=False)
missing_lines

In [ ]:
# Missing values in episodes
missing_episodes = episodes.isna().sum().sort_values(ascending=False)
missing_episodes

In [ ]:
# Distribution of speaking_line values
# We expect this column to indicate whether the row is an actual spoken line.
lines["speaking_line"].value_counts(dropna=False)

In [ ]:
# Inspect the original word_count column.
# We will not trust it blindly because some values may be malformed or incorrectly parsed.
word_count_numeric = pd.to_numeric(lines["word_count"], errors="coerce")
word_count_numeric.describe()

In [ ]:
# Check rows where word_count is not numeric
word_count_numeric = pd.to_numeric(lines["word_count"], errors="coerce")

non_numeric_word_count = lines[word_count_numeric.isna() & lines["word_count"].notna()]

print("Rows with non-numeric word_count:", non_numeric_word_count.shape[0])
non_numeric_word_count[[
    "id",
    "episode_id",
    "raw_character_text",
    "spoken_words",
    "word_count"
]].head(10)

In [ ]:
# Largest original word_count values.
# Extremely high values are suspicious for one dialogue line.
lines.assign(
    word_count_numeric=pd.to_numeric(lines["word_count"], errors="coerce")
).sort_values(
    "word_count_numeric",
    ascending=False
)[[
    "id",
    "episode_id",
    "raw_character_text",
    "spoken_words",
    "word_count",
    "word_count_numeric"
]].head(10)

In [ ]:
# Check rows with embedded CSV fragments inside spoken_words.
# These rows are likely parsing errors and should be removed.
bad_rows = lines["spoken_words"].astype(str).str.contains(
    r"\n\d+,\d+,\d+,",
    regex=True,
    na=False
)

print("Rows with possible embedded CSV fragments:", bad_rows.sum())

lines.loc[bad_rows, [
    "id",
    "episode_id",
    "raw_character_text",
    "spoken_words"
]].head()

In [ ]:
# Initial number of unique characters.
# This explains why using all characters would make the visualizations too cluttered.
print("Unique raw characters:", lines["raw_character_text"].nunique())

lines["raw_character_text"].value_counts().head(20)

In [ ]:
# Check seasons available in the episodes dataset
episodes["season"].value_counts().sort_index()

In [ ]:
# Check whether episode ids from script lines exist in episodes
line_episode_ids = pd.to_numeric(lines["episode_id"], errors="coerce").dropna().astype(int)
episode_ids = pd.to_numeric(episodes["id"], errors="coerce").dropna().astype(int)

missing_episode_ids = set(line_episode_ids.unique()) - set(episode_ids.unique())

print("Episode IDs in script lines not found in episodes:", len(missing_episode_ids))
print(sorted(list(missing_episode_ids))[:20])

## 3. Clean script lines

The raw script file contains non-spoken rows, missing values and some corrupted rows.  
We keep only real spoken lines with valid character and text fields.

In [ ]:
# Work on a copy to keep the raw dataframe untouched
lines_clean = lines.copy()

# Normalize speaking_line
lines_clean["speaking_line"] = (
    lines_clean["speaking_line"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep only actual spoken lines
lines_clean = lines_clean[lines_clean["speaking_line"] == "true"].copy()

# Remove corrupted rows with embedded CSV fragments
bad_rows = lines_clean["spoken_words"].astype(str).str.contains(
    r"\n\d+,\d+,\d+,",
    regex=True,
    na=False
)

lines_clean = lines_clean[~bad_rows].copy()

# Drop rows without essential information
lines_clean = lines_clean.dropna(subset=[
    "episode_id",
    "raw_character_text",
    "spoken_words",
    "normalized_text"
])

# Clean character names
lines_clean["character"] = (
    lines_clean["raw_character_text"]
    .astype(str)
    .str.strip()
)

# Convert episode_id to numeric
lines_clean["episode_id"] = pd.to_numeric(lines_clean["episode_id"], errors="coerce")
lines_clean = lines_clean.dropna(subset=["episode_id"])
lines_clean["episode_id"] = lines_clean["episode_id"].astype(int)

print("Cleaned script lines shape:", lines_clean.shape)

## 4. Recompute word count

The original `word_count` column contains suspicious values, so we recompute it from `normalized_text`.

In [ ]:
# Recompute word count from normalized_text
lines_clean["word_count"] = (
    lines_clean["normalized_text"]
    .astype(str)
    .str.split()
    .str.len()
)

# Remove empty lines
lines_clean = lines_clean[lines_clean["word_count"] > 0].copy()

lines_clean["word_count"].describe()

## 5. Compute sentence count

We approximate the number of sentences by counting sentence-ending punctuation marks.  
If a spoken line has no `.`, `!` or `?`, we count it as one sentence to avoid zero-sentence dialogue lines.

In [ ]:
lines_clean["sentence_count"] = (
    lines_clean["spoken_words"]
    .astype(str)
    .str.count(r"[.!?]+")
)

# If a line has no punctuation, count it as one sentence
lines_clean["sentence_count"] = lines_clean["sentence_count"].clip(lower=1)

lines_clean[["spoken_words", "word_count", "sentence_count"]].head()

## 6. Clean episodes dataset

We clean the episode identifier and keep the metadata columns needed for the analysis.

In [ ]:
episodes_clean = episodes.copy()

# Convert id to numeric
episodes_clean["id"] = pd.to_numeric(episodes_clean["id"], errors="coerce")
episodes_clean = episodes_clean.dropna(subset=["id"])
episodes_clean["id"] = episodes_clean["id"].astype(int)

# Keep useful columns. Add or remove columns here if you need more metadata later.
episode_cols = [
    "id",
    "season",
    "number_in_season",
    "number_in_series",
    "title",
    "original_air_date",
    "imdb_rating",
    "us_viewers_in_millions"
]

episodes_small = episodes_clean[episode_cols].copy()

episodes_small.head()

## 7. Merge script lines with episodes

We join the script lines with the episode metadata using:

`lines_clean.episode_id` → `episodes_small.id`

This adds `season`, `number_in_season`, `title` and other episode information to each dialogue line.

In [ ]:
df = lines_clean.merge(
    episodes_small,
    left_on="episode_id",
    right_on="id",
    how="left"
)

# Remove rows where the episode information was not found
df = df.dropna(subset=["season", "number_in_season", "title"])

# Convert season and episode number to integer
df["season"] = df["season"].astype(int)
df["number_in_season"] = df["number_in_season"].astype(int)

print("Merged dataset shape:", df.shape)
print("Rows without season:", df["season"].isna().sum())
print("Rows without title:", df["title"].isna().sum())

df[["episode_id", "season", "number_in_season", "title", "character", "spoken_words"]].head()

## 8. Keep useful columns

We keep a smaller and cleaner table containing only the variables needed for the visual analytics tool.

In [ ]:
df_clean = df[[
    "episode_id",
    "season",
    "number_in_season",
    "number_in_series",
    "title",
    "original_air_date",
    "character",
    "spoken_words",
    "normalized_text",
    "word_count",
    "sentence_count",
    "imdb_rating",
    "us_viewers_in_millions"
]].copy()

print("Clean dataset shape:", df_clean.shape)
df_clean.head()

## 9. Select the top 10 most relevant characters

Since the dataset contains many characters, we focus on the 10 characters with the highest total number of spoken words.  
This reduces clutter and makes the visualizations easier to interpret.

In [ ]:
top10_characters = (
    df_clean
    .groupby("character")["word_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

print("Top 10 characters:")
for i, character in enumerate(top10_characters, start=1):
    print(f"{i}. {character}")

# Filter the clean dataset to only these top 10 characters
df_top10 = df_clean[df_clean["character"].isin(top10_characters)].copy()

print("\nTop 10 clean dataset shape:", df_top10.shape)

## 10. Create aggregated datasets

For Streamlit and Altair, it is useful to precompute aggregated datasets.  
This keeps the app faster and avoids recalculating the same groupings several times.

In [ ]:
# Total words/sentences per character
character_totals_top10 = (
    df_top10
    .groupby("character", as_index=False)
    .agg(
        total_words=("word_count", "sum"),
        total_sentences=("sentence_count", "sum"),
        total_lines=("spoken_words", "count")
    )
    .sort_values("total_words", ascending=False)
)

character_totals_top10

In [ ]:
# Words/sentences per character and season
character_season_top10 = (
    df_top10
    .groupby(["season", "character"], as_index=False)
    .agg(
        total_words=("word_count", "sum"),
        total_sentences=("sentence_count", "sum"),
        total_lines=("spoken_words", "count")
    )
)

character_season_top10.head()

In [ ]:
# Words/sentences per character and episode
character_episode_top10 = (
    df_top10
    .groupby(
        ["season", "number_in_season", "episode_id", "title", "character"],
        as_index=False
    )
    .agg(
        total_words=("word_count", "sum"),
        total_sentences=("sentence_count", "sum"),
        total_lines=("spoken_words", "count")
    )
)

character_episode_top10.head()

In [ ]:
# Optional: percentage of words per character within each episode.
# This is useful for comparing characters in episodes with different total dialogue volume.
character_episode_top10["episode_total_words"] = (
    character_episode_top10
    .groupby(["season", "number_in_season", "episode_id"])["total_words"]
    .transform("sum")
)

character_episode_top10["word_share"] = (
    character_episode_top10["total_words"] /
    character_episode_top10["episode_total_words"]
)

character_episode_top10.head()

## 11. Final checks

We verify that the final datasets do not contain missing values in the most important columns.

In [ ]:
important_cols = [
    "episode_id",
    "season",
    "number_in_season",
    "title",
    "character",
    "spoken_words",
    "word_count",
    "sentence_count"
]

print("Missing values in important columns:")
df_top10[important_cols].isna().sum()

In [ ]:
print("Final datasets:")
print("df_top10:", df_top10.shape)
print("character_totals_top10:", character_totals_top10.shape)
print("character_season_top10:", character_season_top10.shape)
print("character_episode_top10:", character_episode_top10.shape)

print("\nTop 10 character totals:")
character_totals_top10

## 12. Save clean datasets

The following files will be used later in the Streamlit app and Altair visualizations:

- `simpsons_script_lines_top10_clean.csv`
- `character_totals_top10.csv`
- `character_season_top10.csv`
- `character_episode_top10.csv`

In [ ]:
# Save clean datasets
df_top10.to_csv("simpsons_script_lines_top10_clean.csv", index=False)
character_totals_top10.to_csv("character_totals_top10.csv", index=False)
character_season_top10.to_csv("character_season_top10.csv", index=False)
character_episode_top10.to_csv("character_episode_top10.csv", index=False)

print("Saved files:")
print("- simpsons_script_lines_top10_clean.csv")
print("- character_totals_top10.csv")
print("- character_season_top10.csv")
print("- character_episode_top10.csv")

## 13. Notes for the visualization stage

The visualizations should use the preprocessed CSV files instead of the raw data.  
Recommended usage:

- Use `character_totals_top10.csv` for rankings and overall distributions.
- Use `character_season_top10.csv` for evolution across seasons.
- Use `character_episode_top10.csv` for episode-level comparisons.
- Use `simpsons_script_lines_top10_clean.csv` only if a chart needs line-level detail.

This avoids forcing Streamlit to process unnecessary rows during interaction.